# ECG Heartbeat Classification on the Kaggle MIT-BIH Dataset

This notebook reproduces the heartbeat classifier from **Kachuee, Fazeli & Sarrafzadeh, "ECG Heartbeat Classification: A Deep Transferable Representation" (2018)** on the public Kaggle MIT-BIH dataset, and then asks a simple, practical question for a wearable device:

> Can we make the model **smaller and faster** without losing much accuracy?

We look at three easy-to-explain ideas, measuring **accuracy, model size, and inference time** for each:

1. **Shorter input** — feed the network fewer samples per heartbeat.
2. **A lighter network (MobileNet-style)** — same idea as the paper's CNN, but with cheaper convolutions.
3. **Quantization** — store the trained model using 8-bit integers instead of 32-bit floats.

Everything else follows the original paper: same five heartbeat classes, same single-lead beats, same residual CNN design.

### Use of AI tools
ChatGPT was used to help find the dataset, to explain the MobileNet architecture, and to help write up the report. All code, model training, and results are the author's own.

## 0. Setup

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- settings you can change ----
DATA_DIR        = "."        # folder with mitbih_train.csv / mitbih_test.csv (Kaggle: "ECG Heartbeat Categorization Dataset")
NATIVE_LEN      = 187        # number of samples per beat in the Kaggle CSVs
N_CLASSES       = 5          # heartbeat classes: N, S, V, F, Q
INPUT_LENGTHS   = [187, 140, 96, 64, 48, 32]   # input lengths to test
EPOCHS          = 12         # use ~30 for final results (faster on a GPU)
BATCH_SIZE      = 256
SUBSAMPLE_TRAIN = 20000      # cap training beats for speed; set None to use all
TIMING_REPS     = 100        # how many beats to time when measuring inference speed

try:
    import keras; _kver = keras.__version__
except Exception:
    _kver = getattr(getattr(tf, "keras", None), "__version__", "n/a")
print("TensorFlow", tf.__version__, "| Keras", _kver)

## 1. Load the data

The Kaggle MIT-BIH dataset stores each heartbeat as 187 numbers (the signal) plus a class label (0-4). We load the train/test CSVs. If they aren't found, the notebook makes simple fake data so it still runs, but **use the real Kaggle files for your results**.

In [ ]:
def load_mitbih(data_dir=DATA_DIR):
    tr = os.path.join(data_dir, "mitbih_train.csv")
    te = os.path.join(data_dir, "mitbih_test.csv")
    if os.path.exists(tr) and os.path.exists(te):
        train = pd.read_csv(tr, header=None).values
        test  = pd.read_csv(te, header=None).values
        Xtr, ytr = train[:, :NATIVE_LEN].astype("float32"), train[:, -1].astype("int64")
        Xte, yte = test[:, :NATIVE_LEN].astype("float32"),  test[:, -1].astype("int64")
        print(f"Loaded REAL MIT-BIH: {Xtr.shape[0]} train beats, {Xte.shape[0]} test beats")
        return Xtr, ytr, Xte, yte, True
    print("!! Kaggle CSVs not found -> using FAKE data so the notebook runs.")
    print("!! Download the dataset and set DATA_DIR for real results.")
    def synth(n):
        t = np.linspace(0, 1, NATIVE_LEN)
        y = np.random.randint(0, N_CLASSES, n)
        X = 0.05*np.random.randn(n, NATIVE_LEN).astype("float32")
        for i in range(n):
            X[i] += np.exp(-((t-0.5)**2)/0.002)
            X[i] += 0.4*np.exp(-((t-0.30-0.04*y[i])**2)/0.004)
            X[i] += 0.3*np.exp(-((t-0.70+0.03*y[i])**2)/0.006)
        X = (X - X.min(1, keepdims=True)) / (np.ptp(X, axis=1, keepdims=True) + 1e-6)
        return X.astype("float32"), y
    Xtr, ytr = synth(8000); Xte, yte = synth(2000)
    return Xtr, ytr, Xte, yte, False

Xtr_full, ytr_full, Xte_full, yte_full, IS_REAL = load_mitbih()

# Optionally use a balanced subset of the training beats for speed.
def subsample_balanced(X, y, cap):
    if cap is None or len(X) <= cap: return X, y
    per = cap // N_CLASSES; rng = np.random.default_rng(SEED); idx = []
    for c in range(N_CLASSES):
        ci = np.where(y == c)[0]
        idx.append(rng.choice(ci, size=min(per, len(ci)), replace=False))
    idx = np.concatenate(idx); rng.shuffle(idx)
    return X[idx], y[idx]

Xtr_full, ytr_full = subsample_balanced(Xtr_full, ytr_full, SUBSAMPLE_TRAIN)
print("Training beats used:", Xtr_full.shape[0],
      "| class counts:", np.bincount(ytr_full, minlength=N_CLASSES).tolist())

In [ ]:
# Show one example heartbeat from each class
names = ["N", "S", "V", "F", "Q"]
fig, ax = plt.subplots(1, N_CLASSES, figsize=(15, 2.4), sharey=True)
for c in range(N_CLASSES):
    j = np.where(ytr_full == c)[0]
    if len(j): ax[c].plot(Xtr_full[j[0]]); ax[c].set_title(f"Class {names[c]}")
fig.suptitle("Example heartbeats" + ("" if IS_REAL else "  (FAKE DATA)"))
plt.tight_layout(); plt.show()

## 2. Build the model (same design as the paper)

The paper uses a 1-D convolutional neural network with **residual blocks** (shortcut connections that make deep networks easier to train). We reproduce it: a first convolution, then 5 residual blocks, then two small fully-connected layers and a softmax over the 5 classes.

We also build a **MobileNet-style** version. It is the same shape, but each convolution is split into two cheaper steps (a "depthwise separable" convolution). This is the standard trick for making CNNs small enough to run on phones and watches.

In [ ]:
# ---- the paper's residual CNN ----
def residual_block(x, f=32, k=5):
    shortcut = x
    x = layers.Conv1D(f, k, padding="same")(x); x = layers.ReLU()(x)
    x = layers.Conv1D(f, k, padding="same")(x)
    x = layers.add([x, shortcut]);              x = layers.ReLU()(x)
    return layers.MaxPooling1D(5, strides=2, padding="same")(x)

def build_resnet(input_len, f=32, n_blocks=5, n_classes=N_CLASSES):
    inp = layers.Input((input_len, 1))
    x = layers.Conv1D(f, 5, padding="same")(inp)
    for _ in range(n_blocks): x = residual_block(x, f)
    x = layers.Flatten()(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return models.Model(inp, out, name=f"resnet_L{input_len}")

# ---- the lighter MobileNet-style version ----
def build_mobilenet(input_len, f=32, n_blocks=5, n_classes=N_CLASSES):
    inp = layers.Input((input_len, 1))
    x = layers.Conv1D(f, 5, padding="same")(inp); x = layers.ReLU()(x)
    for _ in range(n_blocks):
        x = layers.SeparableConv1D(f, 5, padding="same")(x); x = layers.ReLU()(x)
        x = layers.MaxPooling1D(5, strides=2, padding="same")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return models.Model(inp, out, name=f"mobilenet_L{input_len}")

### Helper functions: train a model and measure it

For every model we measure three things: **accuracy**, **model size** (kilobytes on disk), and **inference time** (milliseconds to classify one heartbeat).

In [ ]:
def resample_beats(X, target_len):
    """Stretch/shrink each 187-sample beat to a different number of samples."""
    if target_len == X.shape[1]: return X.copy()
    src = np.linspace(0, 1, X.shape[1]); dst = np.linspace(0, 1, target_len)
    return np.stack([np.interp(dst, src, row) for row in X]).astype("float32")

def model_size_kb(model):
    p = f"/tmp/{model.name}.keras"; model.save(p)
    return round(os.path.getsize(p)/1024.0, 1)

def inference_time_ms(model, input_len, reps=TIMING_REPS):
    """Average time to classify one heartbeat, in milliseconds."""
    x = tf.constant(np.random.randn(1, input_len, 1).astype("float32"))
    _ = model(x, training=False)                       # warm-up
    t0 = time.perf_counter()
    for _ in range(reps): _ = model(x, training=False)
    return round((time.perf_counter() - t0) / reps * 1000, 2)

def train_model(model, Xtr, ytr, Xte, yte, epochs=EPOCHS):
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    es = tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True,
                                          monitor="val_accuracy")
    model.fit(Xtr, ytr, validation_data=(Xte, yte), epochs=epochs,
              batch_size=BATCH_SIZE, verbose=0, callbacks=[es])
    return round(model.evaluate(Xte, yte, verbose=0)[1], 4)

## 3. Idea 1 — does a shorter input help?

We train the paper's network using fewer samples per heartbeat (187 down to 32) and record accuracy, size, and inference time at each length.

In [ ]:
rows = []
trained = {}
for L in INPUT_LENGTHS:
    Xtr = resample_beats(Xtr_full, L)[..., None]
    Xte = resample_beats(Xte_full, L)[..., None]
    m = build_resnet(L)
    acc = train_model(m, Xtr, ytr_full, Xte, yte_full)
    rows.append(dict(input_length=L, params=m.count_params(),
                     size_kb=model_size_kb(m),
                     time_ms=inference_time_ms(m, L), accuracy=acc))
    trained[L] = m
    print(f"length {L:3d}: params {m.count_params():6d} | size {rows[-1]['size_kb']:6.1f} KB "
          f"| time {rows[-1]['time_ms']:5.2f} ms | accuracy {acc:.4f}")

sweep = pd.DataFrame(rows)
sweep

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot(sweep.input_length, sweep.accuracy, "o-")
ax[0].set_title("Accuracy vs input length"); ax[0].set_xlabel("input length"); ax[0].set_ylabel("accuracy")
ax[1].plot(sweep.input_length, sweep.time_ms, "o-", color="tab:green")
ax[1].set_title("Inference time vs input length"); ax[1].set_xlabel("input length"); ax[1].set_ylabel("ms per beat")
ax[2].plot(sweep.input_length, sweep.size_kb, "o-", color="tab:orange")
ax[2].set_title("Model size vs input length"); ax[2].set_xlabel("input length"); ax[2].set_ylabel("KB")
plt.tight_layout(); plt.show()

**What this shows.** Accuracy stays high until the input gets very short, but the model size and inference time barely change. So shortening the input is only a small win — most of the size is in the network's weights, not in the input. The next two ideas target the weights directly.

## 4. Idea 2 — a lighter MobileNet-style network

We train the MobileNet-style version at the full input length and compare it to the paper's network.

In [ ]:
L = 187 if 187 in INPUT_LENGTHS else INPUT_LENGTHS[0]
Xtr_L = resample_beats(Xtr_full, L)[..., None]
Xte_L = resample_beats(Xte_full, L)[..., None]

baseline = trained.get(L) or build_resnet(L)
base_acc = float(sweep.loc[sweep.input_length == L, "accuracy"].iloc[0]) if L in trained \
           else train_model(baseline, Xtr_L, ytr_full, Xte_L, yte_full)

mobile = build_mobilenet(L)
mob_acc = train_model(mobile, Xtr_L, ytr_full, Xte_L, yte_full)

print(f"Paper's CNN : {baseline.count_params():6d} params | accuracy {base_acc:.4f}")
print(f"MobileNet   : {mobile.count_params():6d} params | accuracy {mob_acc:.4f}")
print(f"-> MobileNet uses {baseline.count_params()/mobile.count_params():.1f}x fewer parameters")

**What this shows.** The MobileNet-style network has far fewer parameters (a much smaller model) and only loses a little accuracy. This is a much bigger size saving than shortening the input.

## 5. Idea 3 — quantization (8-bit model)

Quantization stores the trained model using 8-bit integers instead of 32-bit floating-point numbers. This makes the saved model roughly 4x smaller. We convert the paper's network with TensorFlow Lite and check that accuracy is preserved.

In [ ]:
def to_int8_tflite(model, X_rep):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    def rep():
        for i in range(min(200, len(X_rep))):
            yield [X_rep[i:i+1].astype("float32")]
    conv.representative_dataset = rep
    return conv.convert()

def tflite_accuracy(blob, X, y):
    itp = tf.lite.Interpreter(model_content=blob); itp.allocate_tensors()
    ind = itp.get_input_details()[0]; outd = itp.get_output_details()[0]
    preds = []
    for i in range(len(X)):
        x = X[i:i+1].astype(ind["dtype"]) if np.issubdtype(ind["dtype"], np.floating) \
            else (X[i:i+1] / ind["quantization"][0] + ind["quantization"][1]).astype(ind["dtype"])
        itp.set_tensor(ind["index"], x); itp.invoke()
        preds.append(int(np.argmax(itp.get_tensor(outd["index"]))))
    return round(float(np.mean(np.array(preds) == y)), 4)

# original (float32) size on disk
float_kb = model_size_kb(baseline)
# quantized (int8) model
blob = to_int8_tflite(baseline, Xtr_L)
int8_kb = round(len(blob) / 1024.0, 1)
int8_acc = tflite_accuracy(blob, Xte_L, yte_full)

print(f"Float32 model : {float_kb:7.1f} KB | accuracy {base_acc:.4f}")
print(f"Int8 model    : {int8_kb:7.1f} KB | accuracy {int8_acc:.4f}")
print(f"-> {float_kb/int8_kb:.1f}x smaller on disk, almost no accuracy change")

## 6. Summary

In [ ]:
summary = pd.DataFrame([
    {"model": "Paper CNN (float32)",     "params": baseline.count_params(), "size_kb": float_kb, "accuracy": base_acc},
    {"model": "Paper CNN (int8)",        "params": baseline.count_params(), "size_kb": int8_kb,  "accuracy": int8_acc},
    {"model": "MobileNet (float32)",     "params": mobile.count_params(),   "size_kb": model_size_kb(mobile), "accuracy": mob_acc},
    {"model": f"Paper CNN, short input (L={int(sweep.input_length.min())})",
     "params": int(sweep.sort_values('input_length').iloc[0].params),
     "size_kb": float(sweep.sort_values('input_length').iloc[0].size_kb),
     "accuracy": float(sweep.sort_values('input_length').iloc[0].accuracy)},
])
print(summary.to_string(index=False))

plt.figure(figsize=(7, 4))
plt.bar(summary.model, summary.size_kb, color="tab:blue")
plt.ylabel("model size (KB)"); plt.title("Model size by approach")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

## 7. Conclusion

Following the original paper, we trained a residual CNN that classifies the five MIT-BIH heartbeat types with high accuracy. To make it suitable for a wearable, we compared three simple ideas:

- **Shorter input** gave only a small reduction in size and time.
- **A MobileNet-style network** cut the number of parameters by several times, losing only a little accuracy.
- **Quantization** made the saved model about 4x smaller with almost no accuracy change.

**Main takeaway:** the best way to shrink the model is to use a lighter network and 8-bit quantization, rather than just shortening the input. These keep the paper's accuracy while making the model small enough to run on a device.

*To run on the real data:* download the Kaggle "ECG Heartbeat Categorization Dataset", set `DATA_DIR` to that folder, raise `EPOCHS` to ~30, and re-run all cells.